## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [56]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 225.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 209.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 234.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 197.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.2.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.2 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [57]:
# Install fundamental build tools
!pip install setuptools wheel -q

In [58]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.2.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.3.3 which is incompatible.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.35.3 which is incompatible.


In [59]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## Question Answering using LLM

#### Downloading and Loading the model

In [60]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [61]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [62]:
user_query_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_query_1)

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

Observation:
Generic, textbook-style answer (early recognition, ABCs). Medically reasonable but not manual-specific — reads like general internet/training knowledge, not Merck Manual content.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [63]:
user_query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_query_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea'

Observation:
Gives some general symptoms (RLQ pain, appetite loss) but answer got cut off at max_tokens=128 before reaching the surgical treatment part of the question.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [64]:
user_query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_query_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

Observation:
Correctly identifies alopecia areata and plausible causes, but again generic/textbook, no mention of specific treatments (truncated before treatment details).

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [65]:
user_query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_query_4)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

Observation:
Reasonable general structure (emergency care → medications) but vague, no specific dosing/protocol grounded in source material.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
user_query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_query_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

Observation:
Sensible first-aid style answer (immobilize, assess shock) but skews toward "field first-aid" rather than a clinical fracture-management protocol from the manual.

### Observations:
Overall, base LLM produces fluent, medical text purely from its pretraining — not from the Manual. This is the core risk: answers are unverifiable against source and frequently truncated mid-thought at 128 tokens.


## Question Answering using LLM with Prompt Engineering

In [67]:
# use the base context as below to improve the LLM answers by giving extra context and role
base_context = "You are a knowledgeable and reliable medical assistant. Answer the following medical question accurately and concisely, covering relevant symptoms, causes, diagnostic steps, and treatment options where applicable. If a definitive answer is not possible, state so clearly rather than guessing."

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [68]:
# update the query with better prompt and add context
user_context_query_1 = base_context+"What is the protocol for managing sepsis in a critical care unit?"
response(user_context_query_1)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition caused by a dysregulated response to an infection. In a critical care unit, managing sepsis involves rapid recognition, assessment, and intervention to prevent progression to septic shock. Here are the general steps for managing sepsis in a critical care unit:\n\n1. Recognition: Look for signs of infection (fever, chills, wound infection) and systemic inflammatory response syndrome (SIRS), which includes fever or hypothermia, tachycardia, tachypnea, and altered mental status.'

Observation:
	More structured (recognition → SIRS criteria), slightly more clinical tone from the "knowledgeable medical assistant" framing.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [69]:
# update the query with better prompt and add context
user_context_query_2 = base_context+"What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_context_query_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is an inflammatory condition of the appendix, a small tube-shaped organ located in the lower right abdomen. Common symptoms include:\n\n1. Sudden and persistent pain in the lower right abdomen, often starting around the navel and moving to the right side.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, which may be low-grade at first but can rise as high as 101°F (38.3°C) or more.\n'

Observation:
	Adds a fever temperature figure (101°F) — sounds more precise, but still not sourced from the manual.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [70]:
user_context_query_3 = base_context+"What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_context_query_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small, round patches from the scalp or other areas of the body. The exact cause of alopecia areata is unknown, but it's believed to be related to a problem with the immune system.\n\nSymptoms:\n- Sudden onset of circular or oval bald spots on the scalp or other parts of the body\n- Hair loss may occur in small patches or involve large areas\n- Hair loss can be temporary"

Observation:
Very similar content to LLM-only, marginal improvement in symptom list.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [71]:
user_context_query_4 = base_context+"What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_context_query_4)

Llama.generate: prefix-match hit


'\n\nA person with a brain injury may require various treatments depending on the severity and location of the injury. Here are some common interventions:\n\n1. Emergency care: For severe brain injuries, immediate medical attention is necessary to prevent further damage or complications. This may include surgery to remove hematomas or decompressing skull fractures, administering medications to manage increased intracranial pressure, and providing supportive care such as maintaining adequate oxygenation and ventilation.\n\n2. Rehabilitation: Brain injury rehabilitation aims to help individuals regain lost skills and improve their overall functioning. This'

Observation:
	Introduces "hematoma removal / decompressing skull fractures" — more clinically specific, though still ungrounded.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [72]:
user_context_query_5 = base_context+"What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_context_query_5)

Llama.generate: prefix-match hit


'\n\nA leg fracture, specifically a hiking-related injury, typically involves a traumatic force applied to the bone, often due to a fall or twisting motion. The necessary precautions and treatment steps are as follows:\n\n1. Assess the situation: Ensure the safety of both the injured person and any companions. If necessary, call for emergency medical assistance.\n2. Immobilize the leg: Use a splint, sling, or other immobilizing device to prevent further damage and provide support. Be careful not to apply excessive pressure on the injury site.\n3. Control bleeding:'

Observation:
	Adds a bleeding-control step not present in the LLM-only answer.


### Observations:
Overall, with Prompt engineering improves tone, structure, and specificity of language, but does not solve the grounding problem — the model is still answering from parametric memory, so accuracy against the actual Manual is still unverified and hallucination risk remains.

In [73]:
params_llm = [
    {"temperature": 0,   "top_p": 0.95, "top_k": 50,  "max_tokens": 128},
    {"temperature": 0.7, "top_p": 0.95, "top_k": 50,  "max_tokens": 128},
    {"temperature": 0,   "top_p": 0.5,  "top_k": 10,  "max_tokens": 128},
    {"temperature": 0.3, "top_p": 0.9,  "top_k": 40,  "max_tokens": 256},
    {"temperature": 1.0, "top_p": 1.0,  "top_k": 100, "max_tokens": 128},
]


# for each combo, call generate_rag_response() for all 5 queries and print responses
for param in params_llm:
    print(f"param: {param}")
    print(response(user_context_query_1, **param))
    print("\n" + "-"*80 + "\n")
    print(response(user_context_query_2, **param))
    print("\n" + "-"*80 + "\n")
    print(response(user_context_query_3, **param))
    print("\n" + "-"*80 + "\n")
    print(response(user_context_query_4, **param))
    print("\n" + "-"*80 + "\n")
    print(response(user_context_query_5, **param))
    print("\n" + "-"*80 + "\n")
    print("\n" + "*"*80 + "\n")

param: {'temperature': 0, 'top_p': 0.95, 'top_k': 50, 'max_tokens': 128}


Llama.generate: prefix-match hit




Sepsis is a life-threatening condition caused by a dysregulated response to an infection. In a critical care unit, managing sepsis involves rapid recognition, assessment, and intervention to prevent progression to septic shock. Here are the general steps for managing sepsis in a critical care unit:

1. Recognition: Look for signs of infection (fever, chills, wound infection) and systemic inflammatory response syndrome (SIRS), which includes fever or hypothermia, tachycardia, tachypnea, and altered mental status.

--------------------------------------------------------------------------------



Llama.generate: prefix-match hit




Appendicitis is an inflammatory condition of the appendix, a small tube-shaped organ located in the lower right abdomen. Common symptoms include:

1. Sudden and persistent pain in the lower right abdomen, often starting around the navel and moving to the right side.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, which may be low-grade at first but can rise as high as 101°F (38.3°C) or more.


--------------------------------------------------------------------------------



...



There are several possible causes of Sudden Patchy Hair Loss (Alopecia Areata), which results in circular or patchy bald spots on the scalp. Some of the common causes include:

1. Autoimmune disease: Alopecia Areata is an autoimmune disorder where the body's immune system attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger the onset of alopecia areata.
3. Genetics: There may be a genetic predisposition to developing this condition.


--------------------------------------------------------------------------------



Llama.generate: prefix-match hit




Treatment for brain injuries can vary depending on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe brain injuries, immediate emergency care is necessary to prevent further damage or death. This may include surgery to remove hematomas (clots) or decompressing skull fractures, administering oxygen, maintaining blood pressure and breathing, and controlling seizures.

2. Medications: Depending on the symptoms, healthcare providers might recommend various medications to manage conditions such as swelling in the brain, pain, infections, or seizures. Some

--------------------------------------------------------------------------------



Llama.generate: prefix-match hit




A leg fracture, specifically a hiking-related injury, typically involves a traumatic force applied to the bone, often due to a fall or twisting motion. Necessary precautions and treatment steps are as follows:

1. Immobilize the injured leg: Use a splint, a sling, or a brace to keep the broken bone from moving and causing further damage. This can be done using items found in the surrounding environment like sticks or clothing until professional medical help arrives.

2. Apply ice packs: Apply ice packs to reduce swelling and pain around the

--------------------------------------------------------------------------------


********************************************************************************



**Observations:**

**params 1 — Baseline (temp=0, top_p=0.95, top_k=50, max_tokens=128)**

Textbook-accurate but generic, and every answer runs out of room before finishing — Q2 stops at a fever number, Q4 cuts off mid-list. Nothing wrong here, just incomplete.

**params 2 — Moderate creativity (temp=0.7, same else)**

Barely different from baseline in substance, mostly just reworded. Q2 gets a slightly sharper description of how the pain migrates, but nothing that changes the actual clinical content. Still truncates at 128 tokens (Q3 cuts off mid-word).

**params 3 — Tight sampling (temp=0, top_p=0.5, top_k=10)**

Identical, word-for-word, to params 1. at temperature=0 the model always picks the top token, so top_p and top_k never get a chance to matter. Clean proof point, not a real variation.

**params 4 — Longer + mild creativity (temp=0.3, top_p=0.9, top_k=40, max_tokens=256)**

This is where it gets interesting — and a little concerning. The extra room lets every answer actually finish, and some of it reads well: Q2 now gives the full appendectomy explanation. But this is also the first place actual hallucination-adjacent content shows up, since there's no retrieval to keep it honest — Q1 states specific numeric targets (MAP >65 mmHg, CVP 8-12 mmHg, urine output >0.5 ml/kg/hr) that sound exactly like something from a real sepsis bundle, but nothing here confirms they're actually sourced from anything — the model is just generating plausible-sounding clinical numbers from memory. That's precisely the risk this section exists to expose.

**params 5 — Max randomness (temp=1.0, top_p=1.0, top_k=100)**

The most unstable of the five. Q1 introduces a lactate threshold out of nowhere. Q4 is the standout moment though — the model actually hedges, saying it "cannot provide a definitive answer without more information," which none of the other params did. That's a good and a bad sign at once: good that high randomness can surface appropriate uncertainty, bad that the other four params answered the exact same ambiguous question with full confidence instead of flagging that ambiguity at all.

**Overall**

Temperature and top_p/top_k mostly just reshuffle wording here — they don't fix the real problem, which is that this is a base LLM with no source to check itself against. The clearest finding is in params 4: giving it more room to talk didn't just produce more complete answers, it produced more specific-sounding answers, including numbers that have no way of being verified.

## Data Preparation for RAG

### Loading the Data

In [74]:
# load text data from pdf at this path /content/medical_diagnosis_manual.pdf
loader = PyMuPDFLoader("/content/medical_diagnosis_manual.pdf")
data = loader.load()

In [75]:
#from google.colab import drive
#drive.mount('/content/drive')

### Data Overview

#### Checking the first 5 pages

In [76]:
# checking first 5 pages
for i in range(5):
    print(f"Page {i+1}")
    print(data[i])

Page 1
page_content='ravi.jntu28@gmail.com
TPG68ZC7FH
This file is meant for personal use by ravi.jntu28@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.' metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/medical_diagnosis_manual.pdf', 'file_path': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-16T05:08:35+00:00', 'trapped': '', 'modDate': 'D:20260816050835Z', 'creationDate': 'D:20120615054440Z', 'page': 0}
Page 2
page_content='ravi.jntu28@gmail.com
TPG68ZC7FH
This file is meant for personal use by ravi.jntu28@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.' metadata={'producer': 'pdf-lib (https://github.com

#### Checking the number of pages

In [77]:
#check data length
len(data)

4114

### Data Chunking

In [78]:
# make this text data into chunks of 1000 with 100 overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(data)

In [79]:
len(texts)

16578

In [80]:
for i in range(5) :
  print(f"\n Page {i+1}")
  print(texts[i])



 Page 1
page_content='ravi.jntu28@gmail.com
TPG68ZC7FH
This file is meant for personal use by ravi.jntu28@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.' metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/medical_diagnosis_manual.pdf', 'file_path': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-16T05:08:35+00:00', 'trapped': '', 'modDate': 'D:20260816050835Z', 'creationDate': 'D:20120615054440Z', 'page': 0}

 Page 2
page_content='ravi.jntu28@gmail.com
TPG68ZC7FH
This file is meant for personal use by ravi.jntu28@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.' metadata={'producer': 'pdf-lib (https://github

### Embedding

In [81]:
import numpy, sentence_transformers
print("numpy:", numpy.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
import sentence_transformers.SentenceTransformer  # will raise cleanly here if still broken

numpy: 2.3.3
sentence_transformers: 5.1.1


In [82]:
#covert these chunks into embeddings
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

embedding_1 = embeddings.embed_query(texts[0].page_content)
embedding_2 = embeddings.embed_query(texts[1].page_content)

print(f"Length of embedding: {len(embedding_1)}")
print(f"Length of embedding: {len(embedding_2)}")


Length of embedding: 384
Length of embedding: 384


**Observation:** My two sample chunks produce embedding vectors of the same fixed dimension, confirming that the embedding model is working consistently across different pieces of text from my manual.

### Vector Database

In [83]:
#create vector store out of this
db = Chroma.from_documents(texts, embeddings)

In [84]:
# do sematic search of the user_query and get top 3 responses
query1 = "What is the protocol for managing sepsis in a critical care unit?"
docs_db = db.similarity_search(query1, k=3)
docs_db

[Document(metadata={'moddate': '2026-08-16T05:08:35+00:00', 'author': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'trapped': '', 'modDate': 'D:20260816050835Z', 'page': 2400, 'subject': '', 'creationDate': 'D:20120615054440Z', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'keywords': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/medical_diagnosis_manual.pdf'}, page_content='16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient rat

### Retriever

In [85]:
#use this vector db as retriver with search as simialrty and top 3 results
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})


In [86]:
# use retriever and get relavant docs for query1
docs = retriever.get_relevant_documents(query=query1)
docs

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 4114, 'format': 'PDF 1.7', 'keywords': '', 'subject': '', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'moddate': '2026-08-16T05:08:35+00:00', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/medical_diagnosis_manual.pdf', 'author': '', 'source': '/content/medical_diagnosis_manual.pdf', 'trapped': '', 'page': 2400, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'modDate': 'D:20260816050835Z'}, page_content='16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient rat

### System and User Prompt Template

In [87]:
qna_system_message = """You are an assistant to a medical professional. Your task is to answer medical questions using only the context provided below, which has been extracted from a trusted medical manual. Do not rely on any outside or prior knowledge. If the context does not contain enough information to answer the question, respond with "I don't have enough information in the provided context to answer that question." Keep your answer factual, clear, and directly grounded in the given context."""

qna_user_message_template = """###Context
Here are some documents that may be relevant to the question:
{context}

###Question
{question}
"""


### Response Function

In [88]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [89]:
# use RAG system to answer this
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1)

Llama.generate: prefix-match hit


'Based on the provided context, the protocol for managing sepsis in a critical care unit includes:\n1. Monitoring physiologic parameters such as temperature, heart rate, respiratory rate, and white blood cell count to identify signs of sepsis (fever or hypothermia, tachycardia or bradycardia, increased or decreased respiratory rate, and leukocytosis or leukopenia with immature forms).\n2. Recognizing severe sepsis when there are signs of organ failure, such as cardiovascular failure (hypotension'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [90]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

Llama.generate: prefix-match hit


"Based on the provided context, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [91]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

Llama.generate: prefix-match hit


'Based on the context provided, the condition being described is likely alopecia areata, a type of nonscarring alopecia characterized by sudden, patchy hair loss. The effective treatments or solutions for addressing this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of alopecia areata is an autoimmune disorder'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [92]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4)

Llama.generate: prefix-match hit


'Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include:\n1. For mild injuries: discharge and observation.\n2. For moderate and severe injuries: optimization of ventilation, oxygenation, and brain perfusion; treatment to lower extremities to increase muscle aerobic capacity, which decreases overall oxygen requirement and eases breathing; supervision while engaging in activity to motivate the patient and make them feel more secure.\n3. Early intervention by rehabilitation specialists for maximal functional recovery, including prevention'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [93]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5)

Llama.generate: prefix-match hit


'Based on the context provided, the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip include:\n1. Rest, ice, and compression: The person should rest the injured leg as much as possible to reduce pain and swelling. Ice should be applied to the affected area to help reduce inflammation, and compression with the use of a thigh sleeve can help minimize swelling.\n2. NSAIDs and analgesics: Nonsteroidal anti-inflammatory drugs (NSAIDs) and analgesics may be prescribed as necessary to manage pain'

### Fine-tuning

In [94]:
# Lets try different combinations of tunable parameters and see what really helps us getting better responses
combos = [
    {"k": 3, "temperature": 0,   "top_p": 0.95, "top_k": 50, "max_tokens": 128},
    {"k": 5, "temperature": 0,   "top_p": 0.95, "top_k": 50, "max_tokens": 128},
    {"k": 3, "temperature": 0.7, "top_p": 0.95, "top_k": 50, "max_tokens": 128},
    {"k": 3, "temperature": 0,   "top_p": 0.5,  "top_k": 10, "max_tokens": 128},
    {"k": 5, "temperature": 0.3, "top_p": 0.85, "top_k": 40, "max_tokens": 256},
]

# for each combo, call generate_rag_response() for all 5 queries and print responses
for combo in combos:
    print(f"Combo: {combo}")
    print(generate_rag_response(user_input_1, **combo))
    print("\n" + "-"*80 + "\n")
    print(generate_rag_response(user_input_2, **combo))
    print("\n" + "-"*80 + "\n")
    print(generate_rag_response(user_input_3, **combo))
    print("\n" + "-"*80 + "\n")
    print(generate_rag_response(user_input_4, **combo))
    print("\n" + "-"*80 + "\n")
    print(generate_rag_response(user_input_5, **combo))
    print("\n" + "-"*80 + "\n")
    print("\n" + "*"*80 + "\n")


Combo: {'k': 3, 'temperature': 0, 'top_p': 0.95, 'top_k': 50, 'max_tokens': 128}


Llama.generate: prefix-match hit


Based on the provided context, the protocol for managing sepsis in a critical care unit includes:
1. Monitoring physiologic parameters such as temperature, heart rate, respiratory rate, and white blood cell count to identify signs of sepsis (fever or hypothermia, tachycardia or bradycardia, increased or decreased respiratory rate, and leukocytosis or leukopenia with immature forms).
2. Recognizing severe sepsis when there are signs of organ failure, such as cardiovascular failure (hypotension

--------------------------------------------------------------------------------



Llama.generate: prefix-match hit


Based on the provided context, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower

--------------------------------------------------------------------------------



...

Based on the provided context, the condition described in the question is likely alopecia areata. The treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of alopecia areata is an autoimmune disorder in which the body's immune system attacks the hair follicles, leading to sudden patchy hair loss. It can also be triggered by stress or other factors. In severe cases, the hair loss may spread to involve larger areas of the scalp or even the entire scalp (alopecia totalis) or the entire body (alopecia universalis). The exact cause of alopecia areata is not fully understood, but it is believed to be related to genetics and environmental factors. It can occur at any age, but it is most common in children and young adults.
Therefore, the effective treatments or solutions for addressing sudden pa

Llama.generate: prefix-match hit


Based on the provided context, the following treatments are recommended for a person who has sustained a physical injury to brain tissue:
1. For mild injuries, discharge and observation.
2. For moderate and severe injuries, optimization of ventilation, oxygenation, and brain perfusion. This may involve the use of lower extremities to increase muscle aerobic capacity and decrease overall oxygen requirement, as well as supervising patients while they engage in activity to motivate them and make them feel more secure.
3. Early intervention by rehabilitation specialists for maximal functional recovery. This includes prevention of secondary disabilities, such as pressure ulcers and joint contractures, prevention of pneumonia, and family education. Rehabilitation may occur in medical facilities or in the patient's home.
4. Emotional care to combat depersonalization and depression that may occur after losing control of the body.
5. Treatments to promote nerve regeneration are under study, inc

Llama.generate: prefix-match hit


Based on the provided context, here's the answer:
A person with a fractured leg, as described in the context, should follow these steps for necessary precautions and treatment:
1. Rest, ice, and compression: Apply ice to the affected area using a thigh sleeve as soon as possible. Compress the injury with a bandage or brace to help reduce swelling and pain.
2. NSAIDs and analgesics: Take nonsteroidal anti-inflammatory drugs (NSAIDs) and analgesics as prescribed by a healthcare professional to manage pain and inflammation.
3. Crutches: Use crutches initially if walking is painful or difficult due to the fracture.
4. Stretching, then strengthening exercises: Once the swelling has decreased and the injury has begun to heal, start with gentle stretching exercises followed by strengthening exercises to improve flexibility and muscle strength around the affected area.
5. Consider potential complications: Be aware of possible complications such as infection or nerve damage, which may require a

Observations:
with different tunable params, responses have wide variety and improved informaiton.

**Combo 1 — Baseline (k=3, temp=0, top_p=0.95, top_k=50, max_tokens=128)**
*Clean and deterministic, but every answer cuts off mid-sentence. Q4 also picks up a stray, off-topic sentence about "muscle aerobic capacity" that doesn't belong — looks like a weak retrieval match sneaking in. Solid floor, but incomplete.*

**Combo 2 — Wider retrieval (k=5, same everything else)**
*This is the one change that actually mattered. Q1 gets the full clinical sequence (cultures → antibiotics → adjust by susceptibility) that baseline missed entirely. And that odd Q4 tangent from Combo 1? Gone — replaced with accurate, on-topic content. More context didn't just add detail, it also crowded out a bad match. Best single-variable result of the five.*

**Combo 3 — Higher temperature (temp=0.7, same retrieval as baseline)**

*Same k=3 chunks, but the model surfaced different facts from them. Q2 now directly answers "no cure via medicine" — something baseline never said despite having the same context available. Interesting, but less reliable than just raising k:*


**Combo 4 — Tight sampling (top_p=0.5, top_k=10, temp still 0)**
*Identical output to baseline, word for word. Makes sense in hindsight — at temperature=0 the model is already picking the single most likely token every time, so top_p and top_k never get a chance to do anything. Worth keeping in the writeup as a clean example that these two params only matter once temperature > 0.*

**Combo 5 — Wider retrieval + mild creativity + more room (k=5, temp=0.3, top_p=0.85, top_k=40, max_tokens=256)**
*The clear winner. Q1 gives a near-complete sepsis workup. Q2 finally delivers the surgical answer in full — appendectomy, laparoscopic vs. open, antibiotics if perforated. Q4 goes deepest of all combos, even mentioning emerging treatments. Q5 becomes the most thorough answer, though it does start pulling in some general wound-care detail that's a bit beyond the original fracture question — a minor side effect of casting a wider net.*

**Bottom line**
Combo 5's configuration delivered the most complete, clinically accurate answers across all 5 queries. k is the lever that actually moves the needle — not temperature, not token limits. Widening retrieval alone fixed both missing content and a bad chunk match. Temperature nudged which facts got pulled from the same context, but unpredictably. And max_tokens=128 was quietly cutting every combo short except the last one — the moment it went to 256, the previously-truncated answers finally landed.


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [95]:
groundedness_rater_system_message  = """You will be given a context and an answer that was generated based on that context. Your task is to rate, on a scale of 1 to 5, how well the answer is grounded in the given context, where 1 means the answer contains claims that are not supported by (or contradict) the context, and 5 means every claim in the answer is fully supported by the context. Provide the numeric rating followed by a brief justification."""


In [96]:
relevance_rater_system_message = """You will be given a question, a context, and an answer that was generated based on that context. Your task is to rate, on a scale of 1 to 5, how relevant the answer is to the question asked, where 1 means the answer does not address the question at all, and 5 means the answer directly and completely addresses the question. Provide the numeric rating followed by a brief justification."""


In [97]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""


In [98]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [99]:
# use ground, rel for user_input_query1
user_input_query1 = "What is the protocol for managing sepsis in a critical care unit?"
ground_query1,rel_query1 = generate_ground_relevance_response(user_input_query1)

print(f"Grounding: {ground_query1}")
print(f"Relevance: {rel_query1}")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Grounding:  Rating: 5

Justification: The answer is fully grounded in the context. The context mentions sepsis as a condition that requires monitoring and treatment in the ICU, and the answer correctly identifies the signs of sepsis and severe sepsis based on the information provided in the context. Additionally, the answer accurately describes the potential manifestations of organ failure in sepsis, which is consistent with the information presented in the context.
Relevance:  Rating: 5

Justification: The answer directly addresses the question by providing specific information about the protocol for managing sepsis in a critical care unit. It mentions monitoring and recognizing signs of sepsis, which aligns with the context's discussion of patient monitoring and testing. Additionally, it identifies severe sepsis and its manifestations as organ failure, which is consistent with the context's definition of severe sepsis. Overall, the answer directly answers the question using informati

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [100]:
user_input_query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground_query2,rel_query2 = generate_ground_relevance_response(user_input_query2)

print(f"Grounding: {ground_query2}")
print(f"Relevance: {rel_query2}")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Grounding:  Rating: 5

Justification: The answer accurately lists the common symptoms for appendicitis as described in the context. It also correctly states that there is no cure for appendicitis and that surgical treatment is necessary, which is consistent with the information provided in the context.
Relevance:  Rating: 5

Justification: The answer directly addresses the question by providing a clear and accurate description of the common symptoms for appendicitis. It also confirms that there is no cure for appendicitis, which was specifically asked about in the question.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [101]:
user_input_query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground_query3,rel_query3 = generate_ground_relevance_response(user_input_query3)

print(f"Grounding: {ground_query3}")
print(f"Relevance: {rel_query3}")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Grounding:  Rating: 5

Justification: The answer is fully grounded in the context as it correctly identifies the condition of sudden patchy hair loss or localized bald spots on the scalp as likely being alopecia areata, and then lists the effective treatments for this condition based on the options provided in the context.
Relevance:  Rating: 5

Justification: The answer directly addresses the question by identifying the condition of sudden patchy hair loss as likely being alopecia areata and providing effective treatments for this condition. The answer sources the information from the context provided, which includes multiple treatment options for alopecia areata.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [102]:
user_input_query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground_query4,rel_query4 = generate_ground_relevance_response(user_input_query4)

print(f"Grounding: {ground_query4}")
print(f"Relevance: {rel_query4}")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Grounding:  Rating: 5

Justification: The context clearly states that for moderate and severe brain injuries, optimization of ventilation, oxygenation, and brain perfusion are recommended treatments. It also mentions the importance of supervising patients during activities to increase muscle aerobic capacity and decrease overall oxygen requirement. For mild injuries, the context states that discharge and observation are the recommended treatments. The answer accurately reflects the information provided in the context.
Relevance:  Rating: 5

Justification: The context clearly outlines the treatments recommended for a person with a brain injury, including discharge and observation for mild injuries and optimization of ventilation, oxygenation, and brain perfusion for moderate to severe injuries. It also mentions the importance of encouraging patients to engage in activities that increase muscle aerobic capacity and the role of early intervention by rehabilitation specialists. The answer 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [103]:
user_input_query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground_query5,rel_query5 = generate_ground_relevance_response(user_input_query5)

print(f"Grounding: {ground_query5}")
print(f"Relevance: {rel_query5}")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Grounding:  I would rate the answer as a 5. The answer accurately summarizes the necessary precautions and treatment steps for a person who has fractured their leg, as outlined in the context. The context mentions "rest, ice, and compression" and the prescription of NSAIDs and analgesics, which are both included in the answer. Therefore, every claim in the answer is fully supported by the context.
Relevance:  Rating: 5

The answer directly addresses the question by listing out the necessary precautions (rest, ice, compression) and treatment steps (NSAIDs and analgesics, use of crutches) for a person who has fractured their leg. The context provided in the text supports these steps, making the answer relevant to the question.


## Actionable Insights and Business Recommendations

1. Retrieval is the single highest-leverage component — deploy RAG, not a standalone LLM.
The base LLM and prompt-engineered LLM produce fluent but unverifiable medical content, essentially generic knowledge dressed up in clinical tone. RAG is the only approach that ties every claim to a traceable source chunk, evidenced by near-perfect groundedness/relevance scores (5/5 on 9 of 10 ratings). For a healthcare use case, this is a compliance and trust requirement, not just a quality improvement — clinicians need to know an answer is Merck Manual–backed, not model-invented.

2. Prompt engineering alone is not a substitute for retrieval.
It helps tone and structure but leaves the core hallucination risk untouched. Recommendation: keep prompt engineering as a complement to RAG (for system/role framing and response formatting), not as an alternative to it.

3. Address truncation — raise max_tokens for production use.
Several base-LLM answers cut off mid-sentence at 128 tokens (e.g., Q2's surgical procedure, Q3's treatment section were never reached). For any deployed assistant, this is a usability failure. Recommend 300–512 tokens minimum for clinical Q&A, with a "continue" mechanism for longer protocols.

4. Build automated groundedness/relevance monitoring into the production pipeline.
The LLM-as-a-judge framework used here should not stay a one-time notebook exercise — operationalize it as a continuous QA layer (sampling live queries, scoring, flagging anything below a groundedness threshold, e.g., <4) before it reaches a clinician-facing UI.

5. Business case: this directly targets the stated problem (information overload, slow retrieval of accurate answers).
A RAG system over the Merck Manual can cut clinician lookup time from manual searching through dense reference text to a sourced, specific answer in seconds — with materially higher factual precision (McBurney's point, drug-class specifics, severity-tiered brain injury treatment) than a generic LLM would ever surface unprompted.

6. Recommended rollout path: pilot the RAG system in a low-stakes, non-diagnostic setting first (e.g., internal clinician reference tool, not patient-facing), expand source documents beyond the single manual, and pair it with the groundedness/relevance judge as a live guardrail before any expansion to higher-stakes clinical decision support.

<font size=6 color='blue'>Power Ahead</font>
___